# Setting up kaggle and downloading the data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
! mkdir ~/.kaggle

In [3]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [4]:
!kaggle competitions download walmart-recruiting-store-sales-forecasting

100% 2.70M/2.70M [00:00<00:00, 91.0MB/s]



In [5]:
! unzip /content/walmart-recruiting-store-sales-forecasting.zip


Archive:  /content/walmart-recruiting-store-sales-forecasting.zip
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           


In [6]:
! unzip /content/walmart-recruiting-store-sales-forecasting.zip
! unzip /content/train.csv.zip
! unzip /content/test.csv.zip
! unzip /content/features.csv.zip
! unzip /content/stores.csv.zip

Archive:  /content/walmart-recruiting-store-sales-forecasting.zip
replace features.csv.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           
Archive:  /content/train.csv.zip
  inflating: train.csv               
Archive:  /content/test.csv.zip
  inflating: test.csv                
Archive:  /content/features.csv.zip
  inflating: features.csv            
unzip:  cannot find or open /content/stores.csv.zip, /content/stores.csv.zip.zip or /content/stores.csv.zip.ZIP.


In [7]:
!pip install mlflow
!pip install dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

# Import libraries

In [8]:
import torch
from torch import nn
from torch.optim import Adam
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import wandb
import torch.nn as nn
import torch.nn.functional as F
from torchsummary import summary
from torchvision import transforms
import torchvision.models as models
import seaborn as sns
import mlflow
import dagshub




device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU
print("Device available: ", device)

Device available:  cpu


# Load and merge the data

In [9]:

train = pd.read_csv('train.csv')
stores = pd.read_csv('stores.csv')
features = pd.read_csv('features.csv')

df = pd.merge(train, stores, on='Store', how='left')
df = pd.merge(df, features, on=['Store', 'Date', 'IsHoliday'], how='left')

df['Date'] = pd.to_datetime(df['Date'])
print(df.head())

   Store  Dept       Date  Weekly_Sales  IsHoliday Type    Size  Temperature  \
0      1     1 2010-02-05      24924.50      False    A  151315        42.31   
1      1     1 2010-02-12      46039.49       True    A  151315        38.51   
2      1     1 2010-02-19      41595.55      False    A  151315        39.93   
3      1     1 2010-02-26      19403.54      False    A  151315        46.63   
4      1     1 2010-03-05      21827.90      False    A  151315        46.50   

   Fuel_Price  MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5  \
0       2.572        NaN        NaN        NaN        NaN        NaN   
1       2.548        NaN        NaN        NaN        NaN        NaN   
2       2.514        NaN        NaN        NaN        NaN        NaN   
3       2.561        NaN        NaN        NaN        NaN        NaN   
4       2.625        NaN        NaN        NaN        NaN        NaN   

          CPI  Unemployment  
0  211.096358         8.106  
1  211.242170         8.10

# WMAE

In [10]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Preprocessing

In [11]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def create_preprocessing_pipeline():
    numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size']
    categorical_features = ['Store', 'Dept', 'Type']
    boolean_features = ['IsHoliday']

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
            ('bool', 'passthrough', boolean_features)
        ])

    return preprocessor

#  training

In [12]:
import mlflow.sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error

dagshub.init(repo_owner="grtsir22", repo_name="walmart-recruiting", mlflow=True)

def train_and_log_model(experiment_name, run_name, model, X_train, y_train, X_val, y_val, is_holiday_val):

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):

        preprocessor = create_preprocessing_pipeline()
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])


        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_val)

        mae = mean_absolute_error(y_val, predictions)

        bias = bias(y_val, predictions)
        custom_wmae = wmae(y_val, predictions, is_holiday_val)

        model_params = model.get_params()
        mlflow.log_params(model_params)
        mlflow.log_param("model_type", type(model).__name__)

        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("bias", bias)
        mlflow.log_metric("WMAE", custom_wmae)

        mlflow.sklearn.log_model(pipeline, "model_pipeline")

        print(f"[{run_name}] logged! WMAE: {custom_wmae:.4f}")

        return pipeline

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=bc1331cc-8a27-4a82-b5cd-3cd50d6eb8b5&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=06a8c8085df7f0c5f00b66611c595e68622c5c4404a08c45d715097cb605f10c




Accessing as grtsir22

Initialized MLflow to track repo "grtsir22/walmart-recruiting"

Repository grtsir22/walmart-recruiting initialized!

# Extract features and split

In [13]:
def prepare_and_split_data(df, val_split_date="2012-03-26"):

    df = df.copy()
    df['Year'] = df['Date'].dt.isocalendar().year
    df['Month'] = df['Date'].dt.month
    df['Week'] = df['Date'].dt.isocalendar().week

    df = df.sort_values(by='Date')

    train_df = df[df['Date'] < val_split_date]
    val_df = df[df['Date'] >= val_split_date]

    X_train = train_df.drop(columns=['Weekly_Sales', 'Date'])
    y_train = train_df['Weekly_Sales']

    X_val = val_df.drop(columns=['Weekly_Sales', 'Date'])
    y_val = val_df['Weekly_Sales']

    is_holiday_val = X_val['IsHoliday'].values

    return X_train, y_train, X_val, y_val, is_holiday_val

X_train, y_train, X_val, y_val, is_holiday_val = prepare_and_split_data(df)

# preprocessing

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def create_preprocessing_pipeline():
    markdown_features = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
    numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size'] + markdown_features
    categorical_features = ['Store', 'Dept', 'Type', 'Year', 'Month', 'Week']
    boolean_features = ['IsHoliday']

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
            ('bool', 'passthrough', boolean_features)
        ])

    return preprocessor

# SARIMAX

In [15]:
!pip install pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.6 MB/s eta 0:00:00


In [17]:
import pmdarima as pm
import pandas as pd
import numpy as np

ts_df = df[(df['Store'] == 1) & (df['Dept'] == 1)].copy()

ts_df = ts_df.sort_values('Date')

val_split_date = "2012-05-01"
train_ts = ts_df[ts_df['Date'] < val_split_date]
val_ts = ts_df[ts_df['Date'] >= val_split_date]

exog_features = ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

y_train_ts = train_ts['Weekly_Sales']
X_train_ts = train_ts[exog_features]

y_val_ts = val_ts['Weekly_Sales']
X_val_ts = val_ts[exog_features]

In [18]:
import pandas as pd
import numpy as np


exog_features = ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

if ts_df['IsHoliday'].dtype == bool:
    ts_df['IsHoliday'] = ts_df['IsHoliday'].astype(int)

ts_df[exog_features] = ts_df[exog_features].ffill().bfill().fillna(0)

ts_df[exog_features] = ts_df[exog_features].astype(float)
ts_df['Weekly_Sales'] = ts_df['Weekly_Sales'].astype(float)

val_split_date = "2012-03-26"
train_ts = ts_df[ts_df['Date'] < val_split_date]
val_ts = ts_df[ts_df['Date'] >= val_split_date]

y_train_ts = train_ts['Weekly_Sales']
X_train_ts = train_ts[exog_features]

y_val_ts = val_ts['Weekly_Sales']
X_val_ts = val_ts[exog_features]

In [19]:

def train_sarimax_baseline(experiment_name, run_name, y_train, X_train, y_val, X_val):
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):
        print("Fitting Auto ARIMA... this may take a minute.")
        model = pm.auto_arima(
            y_train,
            X=X_train,
            seasonal=True,
            m=52,
            suppress_warnings=True,
            stepwise=True,
            trace=True
        )

        predictions = model.predict(n_periods=len(y_val), X=X_val)

        mae = np.mean(np.abs(y_val - predictions))
        is_holiday_val = X_val['IsHoliday'].values
        custom_wmae = wmae(y_val.values, predictions.values, is_holiday_val)

        mlflow.log_param("model_type", "SARIMAX")
        mlflow.log_param("order", model.order)
        mlflow.log_param("seasonal_order", model.seasonal_order)

        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("WMAE", custom_wmae)

        print(f"[{run_name}] SARIMAX logged! WMAE: {custom_wmae:.4f}")
        return model

sarima_model = train_sarimax_baseline(
    experiment_name="SARIMAX",
    run_name="SARIMAX_Store1_Dept1",
    y_train=y_train_ts,
    X_train=X_train_ts,
    y_val=y_val_ts,
    X_val=X_val_ts
)

2026/07/12 13:50:30 INFO mlflow.tracking.fluent: Experiment with name 'SARIMAX' does not exist. Creating a new experiment.


Fitting Auto ARIMA... this may take a minute.
Performing stepwise search to minimize aic
 ARIMA(2,0,2)(1,0,1)[52] intercept   : AIC=inf, Time=17.07 sec
 ARIMA(0,0,0)(0,0,0)[52] intercept   : AIC=2369.830, Time=0.03 sec
 ARIMA(1,0,0)(1,0,0)[52] intercept   : AIC=2330.850, Time=10.30 sec
 ARIMA(0,0,1)(0,0,1)[52] intercept   : AIC=2323.411, Time=8.00 sec
 ARIMA(0,0,0)(0,0,0)[52]             : AIC=2367.834, Time=0.23 sec
 ARIMA(0,0,1)(0,0,0)[52] intercept   : AIC=2340.288, Time=2.24 sec
 ARIMA(0,0,1)(1,0,1)[52] intercept   : AIC=2327.913, Time=8.43 sec
 ARIMA(0,0,1)(0,0,2)[52] intercept   : AIC=2328.177, Time=43.39 sec
 ARIMA(0,0,1)(1,0,0)[52] intercept   : AIC=2320.277, Time=7.22 sec
 ARIMA(0,0,1)(2,0,0)[52] intercept   : AIC=2328.953, Time=61.57 sec
 ARIMA(0,0,1)(2,0,1)[52] intercept   : AIC=inf, Time=66.98 sec
 ARIMA(0,0,0)(1,0,0)[52] intercept   : AIC=2353.582, Time=9.52 sec
 ARIMA(1,0,1)(1,0,0)[52] intercept   : AIC=2327.931, Time=5.78 sec
 ARIMA(0,0,2)(1,0,0)[52] intercept   : AIC=23

In [25]:
sarima_model = train_sarimax_baseline(
    experiment_name="time_series",
    run_name="SARIMAX_Store1_Dept1",
    y_train=y_train_ts,
    X_train=X_train_ts,
    y_val=y_val_ts,
    X_val=X_val_ts
)

Fitting Auto ARIMA... this may take a minute.
Performing stepwise search to minimize aic
 ARIMA(2,0,2)(1,0,1)[52] intercept   : AIC=inf, Time=15.22 sec
 ARIMA(0,0,0)(0,0,0)[52] intercept   : AIC=2369.830, Time=0.02 sec
 ARIMA(1,0,0)(1,0,0)[52] intercept   : AIC=2330.850, Time=8.65 sec
 ARIMA(0,0,1)(0,0,1)[52] intercept   : AIC=2323.411, Time=8.01 sec
 ARIMA(0,0,0)(0,0,0)[52]             : AIC=2367.834, Time=0.15 sec
 ARIMA(0,0,1)(0,0,0)[52] intercept   : AIC=2340.288, Time=0.25 sec
 ARIMA(0,0,1)(1,0,1)[52] intercept   : AIC=2327.913, Time=10.89 sec
 ARIMA(0,0,1)(0,0,2)[52] intercept   : AIC=2328.177, Time=34.76 sec
 ARIMA(0,0,1)(1,0,0)[52] intercept   : AIC=2320.277, Time=6.85 sec
 ARIMA(0,0,1)(2,0,0)[52] intercept   : AIC=2328.953, Time=43.02 sec
 ARIMA(0,0,1)(2,0,1)[52] intercept   : AIC=inf, Time=48.25 sec
 ARIMA(0,0,0)(1,0,0)[52] intercept   : AIC=2353.582, Time=5.62 sec
 ARIMA(1,0,1)(1,0,0)[52] intercept   : AIC=2327.931, Time=10.12 sec
 ARIMA(0,0,2)(1,0,0)[52] intercept   : AIC=2

# Prophet

In [20]:
!pip install prophet

In [21]:

prophet_train = train_ts.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
prophet_val = val_ts.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})

exog_features = ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

In [22]:
import mlflow
import mlflow.prophet
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

def train_prophet_baseline(experiment_name, run_name, train_df, val_df, exog_features):
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False
        )

        for col in exog_features:
            model.add_regressor(col)

        print("Fitting Prophet...")
        model.fit(train_df)

        future_df = val_df[['ds'] + exog_features]
        forecast = model.predict(future_df)

        predictions = forecast['yhat'].values
        y_true = val_df['y'].values
        is_holiday_val = val_df['IsHoliday'].values

        mae = mean_absolute_error(y_true, predictions)
        custom_wmae = wmae(y_true, predictions, is_holiday_val)

        mlflow.log_param("model_type", "Prophet")
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("WMAE", custom_wmae)

        mlflow.prophet.log_model(model, "prophet_model")

        print(f"[{run_name}] Prophet logged! WMAE: {custom_wmae:.4f}")
        return model, forecast

prophet_model, prophet_forecast = train_prophet_baseline(
    experiment_name="time_series",
    run_name="Prophet_Store1_Dept1",
    train_df=prophet_train,
    val_df=prophet_val,
    exog_features=exog_features
)

Fitting Prophet...


2026/07/12 13:57:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[Prophet_Store1_Dept1] Prophet logged! WMAE: 3654.8721
🏃 View run Prophet_Store1_Dept1 at: https://dagshub.com/grtsir22/walmart-recruiting.mlflow/#/experiments/1/runs/0ca267d35b144955921603203f71af0a
🧪 View experiment at: https://dagshub.com/grtsir22/walmart-recruiting.mlflow/#/experiments/1


# We won't really have features in the test set

In [23]:
prophet_train = train_ts.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
prophet_val = val_ts.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})

exog_features = ['IsHoliday']

In [24]:
import mlflow
import mlflow.prophet
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

def train_prophet_baseline(experiment_name, run_name, train_df, val_df, exog_features):
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False
        )

        for col in exog_features:
            model.add_regressor(col)

        print("Fitting Prophet...")
        model.fit(train_df)

        future_df = val_df[['ds'] + exog_features]
        forecast = model.predict(future_df)

        predictions = forecast['yhat'].values
        y_true = val_df['y'].values
        is_holiday_val = val_df['IsHoliday'].values

        mae = mean_absolute_error(y_true, predictions)
        custom_wmae = wmae(y_true, predictions, is_holiday_val)

        mlflow.log_param("model_type", "Prophet")
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("WMAE", custom_wmae)

        mlflow.prophet.log_model(model, "prophet_model")

        print(f"[{run_name}] Prophet logged! WMAE: {custom_wmae:.4f}")
        return model, forecast

prophet_model, prophet_forecast = train_prophet_baseline(
    experiment_name="time_series",
    run_name="Prophet-final_Store1_Dept1",
    train_df=prophet_train,
    val_df=prophet_val,
    exog_features=exog_features
)

Fitting Prophet...


2026/07/12 13:57:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[Prophet-final_Store1_Dept1] Prophet logged! WMAE: 3127.9375
🏃 View run Prophet-final_Store1_Dept1 at: https://dagshub.com/grtsir22/walmart-recruiting.mlflow/#/experiments/1/runs/d777d10c8c8e40949cb30999fce7bba1
🧪 View experiment at: https://dagshub.com/grtsir22/walmart-recruiting.mlflow/#/experiments/1
